# GA Selection Operators (Multi-Vehicle Routing)

Este notebook demonstra os **4 operadores de seleção** usados na implementação do GA deste projeto:

1. **Roulette Selection** — probabilidade proporcional ao inverso do fitness
2. **Rank Selection** — probabilidade proporcional ao ranking (não ao valor bruto de fitness)
3. **Stochastic Universal Sampling (SUS)** — variante de menor variância da roleta, usando 2 ponteiros equidistantes
4. **Tournament Selection** — amostra K competidores aleatórios e elege o melhor

**Interface pública:** todos os operadores implementam:
```python
select_parents(
    population: Population,
    evaluated_population: list[FleetRouteInfo],
    fitness_function: Callable[[FleetRouteInfo], float],
) -> tuple[Individual, Individual]
```

- `population[i]` é o indivíduo (lista de rotas com `RouteNode`)
- `evaluated_population[i]` é o `FleetRouteInfo` com as métricas calculadas para `population[i]`
- `fitness_function` recebe um `FleetRouteInfo` e retorna um `float` — **menor = melhor**

Cada seção segue a estrutura:
- **Teoria** — descrição do algoritmo
- **Bare-metal** — implementação explícita em listas puras para visualizar a lógica
- **Uso com a classe do repositório** — chamada à estratégia concreta da infraestrutura

In [1]:
#!pip install ipykernel
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

from src.infrastructure.genetic_algorithm.selection import (
    RoulleteSelectionStrategy,
    RankSelectionStrategy,
    StochasticUniversalSamplingSelectionStrategy,
    TournamentSelectionStrategy,
)
from src.domain.models.graph import RouteNode
from src.domain.models.route import FleetRouteInfo

# --- Nós de destino ---
origin = RouteNode(name="Origin", node_id=0, coords=(0.0, 0.0))
A = RouteNode(name="A", node_id=1, coords=(1.0, 0.0))
B = RouteNode(name="B", node_id=2, coords=(2.0, 0.0))
C = RouteNode(name="C", node_id=3, coords=(2.0, 1.0))
D = RouteNode(name="D", node_id=4, coords=(1.0, 1.0))
E = RouteNode(name="E", node_id=5, coords=(0.0, 1.0))
F = RouteNode(name="F", node_id=6, coords=(0.0, 0.5))
G = RouteNode(name="G", node_id=7, coords=(1.5, 1.5))
H = RouteNode(name="H", node_id=8, coords=(2.5, 0.5))

# --- Mini população: 5 indivíduos com ordens de visita diferentes ---
population = [
    [[origin, A, B, C, D, E, F, G, H]],  # ind 0 — custo alto
    [[origin, E, A, C, H, B, D, G, F]],  # ind 1 — melhor custo
    [[origin, B, D, F, H, G, E, C, A]],  # ind 2
    [[origin, H, G, F, E, D, C, B, A]],  # ind 3 — pior custo
    [[origin, C, A, E, G, B, D, F, H]],  # ind 4
]

# --- evaluated_population: FleetRouteInfo com custo pré-calculado para cada indivíduo ---
# Construímos diretamente com total_cost para simplificar o exemplo
# (em produção esses valores são calculados via adjacency matrix)
evaluated_population = [
    FleetRouteInfo(total_cost=320.0, total_eta=320.0, total_length=0.0),  # ind 0
    FleetRouteInfo(total_cost=180.0, total_eta=180.0, total_length=0.0),  # ind 1 — melhor
    FleetRouteInfo(total_cost=270.0, total_eta=270.0, total_length=0.0),  # ind 2
    FleetRouteInfo(total_cost=410.0, total_eta=410.0, total_length=0.0),  # ind 3 — pior
    FleetRouteInfo(total_cost=230.0, total_eta=230.0, total_length=0.0),  # ind 4
]

# --- Fitness function: menor total_cost = melhor ---
def fitness_fn(info: FleetRouteInfo) -> float:
    return info.total_cost or 1e-6

def names(individual):
    return [[node.name for node in route] for route in individual]

print("População (ordem de visita | custo):")
for i, (ind, ev) in enumerate(zip(population, evaluated_population)):
    print(f"  Ind {i}: {names(ind)}  → custo = {ev.total_cost:.0f}")

População (ordem de visita | custo):
  Ind 0: [['Origin', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']]  → custo = 320
  Ind 1: [['Origin', 'E', 'A', 'C', 'H', 'B', 'D', 'G', 'F']]  → custo = 180
  Ind 2: [['Origin', 'B', 'D', 'F', 'H', 'G', 'E', 'C', 'A']]  → custo = 270
  Ind 3: [['Origin', 'H', 'G', 'F', 'E', 'D', 'C', 'B', 'A']]  → custo = 410
  Ind 4: [['Origin', 'C', 'A', 'E', 'G', 'B', 'D', 'F', 'H']]  → custo = 230


## 1) Roulette Selection (Fitness Proportionate Selection)

A **Roulette Selection** simula uma roleta onde cada indivíduo ocupa uma fatia proporcional à sua probabilidade de seleção.

Como o GA resolve um problema de **minimização** (menor custo = melhor), a probabilidade é calculada como o **inverso do fitness**:

$$P_i = \frac{1/f_i}{\sum_{j} 1/f_j}$$

Assim, indivíduos com **menor custo** recebem **fatias maiores** na roleta.

### Processo
1. Calcula `weight_i = 1 / max(fitness_i, ε)` para cada indivíduo (ε = 1e-9 evita divisão por zero)
2. Normaliza os pesos para obter probabilidades
3. Sorteia 2 pais usando `random.choices(population, weights=...)`

### Vantagem
Simples e intuitivo. Indivíduos com ótimo fitness dominam a seleção.

### Desvantagem
Alta variância: se um indivíduo é muito superior aos demais, ele pode monopolizar a seleção, causando **convergência prematura**.

In [2]:
# A. Implementação "Bare-metal"
import random

# Fitness de cada indivíduo (custo — menor = melhor)
sim_fitness = [320.0, 180.0, 270.0, 410.0, 230.0]

# Pesos = inverso do fitness (indivíduo mais barato → fatia maior)
weights = [1 / f for f in sim_fitness]
total   = sum(weights)
probs   = [w / total for w in weights]

print("Ind | Custo  | Peso inv.  | Prob. seleção")
print("-" * 45)
for i, (cost, w, p) in enumerate(zip(sim_fitness, weights, probs)):
    bar = "█" * int(p * 40)
    print(f"  {i} | {cost:5.0f}  | {w:.6f}  | {p:.1%}  {bar}")

# Simulação de 1000 seleções para mostrar a distribuição empírica
counts = {i: 0 for i in range(len(sim_fitness))}
for _ in range(1000):
    selected = random.choices(range(len(sim_fitness)), weights=probs, k=1)[0]
    counts[selected] += 1

print("\nDistribuição empírica (1000 sorteios):")
for i, c in counts.items():
    print(f"  Ind {i} (custo={sim_fitness[i]:.0f}): {c} vezes ({c/10:.1f}%)")

Ind | Custo  | Peso inv.  | Prob. seleção
---------------------------------------------
  0 |   320  | 0.003125  | 16.3%  ██████
  1 |   180  | 0.005556  | 29.0%  ███████████
  2 |   270  | 0.003704  | 19.3%  ███████
  3 |   410  | 0.002439  | 12.7%  █████
  4 |   230  | 0.004348  | 22.7%  █████████

Distribuição empírica (1000 sorteios):
  Ind 0 (custo=320): 166 vezes (16.6%)
  Ind 1 (custo=180): 292 vezes (29.2%)
  Ind 2 (custo=270): 174 vezes (17.4%)
  Ind 3 (custo=410): 130 vezes (13.0%)
  Ind 4 (custo=230): 238 vezes (23.8%)


---
### Uso com `RoulleteSelectionStrategy` no repositório

A classe lida com `RouteNode` objects dentro de `FleetRouteInfo` e usa `numpy` para normalizar as probabilidades.

In [5]:
# B. Usando a classe de domínio do projeto
roulette = RoulleteSelectionStrategy()

parent1, parent2 = roulette.select_parents(population, evaluated_population, fitness_fn)
print("Parent 1:", names(parent1))
print("Parent 2:", names(parent2))

Parent 1: [['Origin', 'C', 'A', 'E', 'G', 'B', 'D', 'F', 'H']]
Parent 2: [['Origin', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']]


## 2) Rank Selection

O problema da Roulette Selection é que ela é sensível à escala do fitness: se um indivíduo tem custo `1` e todos os outros têm custo `1000`, ele domina completamente a seleção.

O **Rank Selection** resolve isso **descartando os valores brutos de fitness** e trabalhando apenas com a *posição no ranking*:

1. Ordena a população do melhor ao pior (menor custo = rank 1)
2. Atribui pesos baseados no rank: o melhor recebe peso `N`, o segundo melhor `N-1`, ..., o pior recebe peso `1`
3. Sorteia 2 pais usando esses pesos

$$\text{weight}_i = N - \text{rank}_i$$

### Vantagem
- Elimina a dominância de um único indivíduo muito superior
- Todo indivíduo, mesmo o pior, tem **weight ≥ 1** — garante diversidade genética

### Desvantagem
- Perde informação sobre a magnitude real das diferenças de fitness
- Pressão de seleção é sempre constante (independente do quão longe o melhor está dos demais)

In [9]:
# A. Implementação "Bare-metal"
sim_fitness = [320.0, 180.0, 270.0, 410.0, 230.0]
n = len(sim_fitness)

# Ordena índices do melhor (menor custo) ao pior (maior custo)
ranked_indexes = sorted(range(n), key=lambda i: sim_fitness[i])

# Atribui pesos: rank 0 (melhor) → peso N, rank N-1 (pior) → peso 1
rank_weights     = [0] * n
roulette_weights = [1 / f for f in sim_fitness]
roulette_total   = sum(roulette_weights)

for rank, idx in enumerate(ranked_indexes):
    rank_weights[idx] = n - rank

rank_total = sum(rank_weights)

print("Ind | Custo | Rank | Peso Rank | Prob Rank | Prob Roleta")
print("-" * 62)
for i in range(n):
    prob_rank     = rank_weights[i] / rank_total
    prob_roulette = roulette_weights[i] / roulette_total
    r = ranked_indexes.index(i) + 1
    print(f"  {i} | {sim_fitness[i]:5.0f} | {r:4} | {rank_weights[i]:9} | {prob_rank:9.1%} | {prob_roulette:.1%}")

worst = 3  # Ind 3 (custo=410) é o pior
print(f"\n→ Com custos similares (180–410), o pior indivíduo tem:")
print(f"    Rank:   {rank_weights[worst]/rank_total:.1%}")
print(f"    Roleta: {roulette_weights[worst]/roulette_total:.1%}")
print("  (nesse caso a roleta e o rank ficam próximos)")

# --- Cenário extremo: mostra onde o Rank se destaca ---
print("\n--- Cenário extremo (1 superdominante, resto mediocre) ---")
extreme_fitness = [5.0, 1000.0, 1000.0, 1000.0, 1000.0]   # Ind 0 é muito melhor

ext_rank_weights     = [0] * n
ext_roulette_weights = [1 / f for f in extreme_fitness]
ext_roulette_total   = sum(ext_roulette_weights)

ext_ranked = sorted(range(n), key=lambda i: extreme_fitness[i])
for rank, idx in enumerate(ext_ranked):
    ext_rank_weights[idx] = n - rank
ext_rank_total = sum(ext_rank_weights)

print("Ind | Custo  | Prob Rank | Prob Roleta")
print("-" * 42)
for i in range(n):
    print(f"  {i} | {extreme_fitness[i]:6.0f} | {ext_rank_weights[i]/ext_rank_total:9.1%} | {ext_roulette_weights[i]/ext_roulette_total:.1%}")

print(f"\n→ Na roleta, os piores (custo=1000) ficam com ~0.1% cada — quase eliminados.")
print(f"  No rank, o pior ainda tem {ext_rank_weights[ext_ranked[-1]]/ext_rank_total:.1%} — mantém diversidade.")

Ind | Custo | Rank | Peso Rank | Prob Rank | Prob Roleta
--------------------------------------------------------------
  0 |   320 |    4 |         2 |     13.3% | 16.3%
  1 |   180 |    1 |         5 |     33.3% | 29.0%
  2 |   270 |    3 |         3 |     20.0% | 19.3%
  3 |   410 |    5 |         1 |      6.7% | 12.7%
  4 |   230 |    2 |         4 |     26.7% | 22.7%

→ Com custos similares (180–410), o pior indivíduo tem:
    Rank:   6.7%
    Roleta: 12.7%
  (nesse caso a roleta e o rank ficam próximos)

--- Cenário extremo (1 superdominante, resto mediocre) ---
Ind | Custo  | Prob Rank | Prob Roleta
------------------------------------------
  0 |      5 |     33.3% | 98.0%
  1 |   1000 |     26.7% | 0.5%
  2 |   1000 |     20.0% | 0.5%
  3 |   1000 |     13.3% | 0.5%
  4 |   1000 |      6.7% | 0.5%

→ Na roleta, os piores (custo=1000) ficam com ~0.1% cada — quase eliminados.
  No rank, o pior ainda tem 6.7% — mantém diversidade.


---
### Uso com `RankSelectionStrategy` no repositório

A classe ordena a população pelo resultado de `fitness_function(evaluated_population[i])` e atribui pesos inteiros inversamente proporcionais ao rank.

In [11]:
# B. Usando a classe de domínio do projeto
rank = RankSelectionStrategy()

parent1, parent2 = rank.select_parents(population, evaluated_population, fitness_fn)
print("Parent 1:", names(parent1))
print("Parent 2:", names(parent2))

Parent 1: [['Origin', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']]
Parent 2: [['Origin', 'B', 'D', 'F', 'H', 'G', 'E', 'C', 'A']]


## 3) Stochastic Universal Sampling (SUS)

O **SUS** é uma variante da Roulette Selection projetada para **reduzir a variância** no processo de seleção.

### O problema da Roulette dupla
Na Roulette convencional, para selecionar 2 pais, giramos a roleta **duas vezes** independentemente. Isso significa que por azar podemos selecionar o mesmo indivíduo ótimo duas vezes — ou nunca selecionar um indivíduo mediano que deveria aparecer.

### Solução do SUS: ponteiros equidistantes
Em vez de girar a roleta duas vezes, o SUS coloca **2 ponteiros igualmente espaçados** e gira apenas uma vez:

```
Roleta (pesos acumulados):
|--Ind0----|---Ind1------|--Ind2---|-------Ind3------|--Ind4----|
0         w0          w0+w1    ...                          total

step = total / 2
Ponteiro 1: start       (aleatório entre 0 e step)
Ponteiro 2: start + step
```

Percorremos a distribuição cumulativa e encontramos qual indivíduo cada ponteiro aponta.

### Processo
1. Calcula `weight_i = 1 / fitness_i`
2. `total = sum(weights)`, `step = total / 2`
3. Sorteia `start` aleatório em `[0, step)`
4. Para cada ponteiro em `[start, start + step]`: percorre a soma cumulativa e retorna o índice correspondente

### Vantagem
- **Menor variância** que a Roulette: garante uma cobertura mais uniforme da população
- Indivíduos com fatia maior que `step` têm garantia de ser selecionados pelo menos uma vez

In [12]:
# A. Implementação "Bare-metal"
import random

sim_fitness = [320.0, 180.0, 270.0, 410.0, 230.0]

# 1. Pesos = inverso do fitness
weights = [1 / f for f in sim_fitness]
total   = sum(weights)

# 2. Passo entre os dois ponteiros
step = total / 2

# 3. Ponto de partida aleatório (fixamos para o exemplo ser reproduzível)
random.seed(42)
start = random.uniform(0, step)
pointers = [start, start + step]

# 4. Distribuição cumulativa
cumulative = []
acc = 0.0
for w in weights:
    acc += w
    cumulative.append(acc)

print(f"Pesos (1/fitness): {[f'{w:.6f}' for w in weights]}")
print(f"Total: {total:.6f}  |  Step: {step:.6f}")
print(f"Start: {start:.6f}  |  Ponteiros: {[f'{p:.6f}' for p in pointers]}\n")

print("Ind | Custo | Peso Acum.  | Ponteiro 1 cobre? | Ponteiro 2 cobre?")
print("-" * 68)
selected = []
for pointer in pointers:
    for i, cum in enumerate(cumulative):
        if cum >= pointer:
            selected.append(i)
            break

for i, (cum, w) in enumerate(zip(cumulative, weights)):
    p1 = "✓ selecionado" if selected[0] == i else ""
    p2 = "✓ selecionado" if selected[1] == i else ""
    print(f"  {i} | {sim_fitness[i]:5.0f} | {cum:.6f}   | {p1:<16} | {p2}")

print(f"\nPais selecionados: Ind {selected[0]} (custo={sim_fitness[selected[0]]:.0f}) e Ind {selected[1]} (custo={sim_fitness[selected[1]]:.0f})")

Pesos (1/fitness): ['0.003125', '0.005556', '0.003704', '0.002439', '0.004348']
Total: 0.019171  |  Step: 0.009586
Start: 0.006129  |  Ponteiros: ['0.006129', '0.015715']

Ind | Custo | Peso Acum.  | Ponteiro 1 cobre? | Ponteiro 2 cobre?
--------------------------------------------------------------------
  0 |   320 | 0.003125   |                  | 
  1 |   180 | 0.008681   | ✓ selecionado    | 
  2 |   270 | 0.012384   |                  | 
  3 |   410 | 0.014823   |                  | 
  4 |   230 | 0.019171   |                  | ✓ selecionado

Pais selecionados: Ind 1 (custo=180) e Ind 4 (custo=230)


---
### Uso com `StochasticUniversalSamplingSelectionStrategy` no repositório

A classe encapsula a lógica de ponteiros em `_select_index()`, que percorre linearmente a distribuição cumulativa para cada ponteiro.

In [13]:
# B. Usando a classe de domínio do projeto
sus = StochasticUniversalSamplingSelectionStrategy()

parent1, parent2 = sus.select_parents(population, evaluated_population, fitness_fn)
print("Parent 1:", names(parent1))
print("Parent 2:", names(parent2))

Parent 1: [['Origin', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']]
Parent 2: [['Origin', 'B', 'D', 'F', 'H', 'G', 'E', 'C', 'A']]


## 4) Tournament Selection

O **Tournament Selection** não usa a distribuição global de fitness. Em vez disso, organiza **micro-competições** locais:

1. Sorteia `K` competidores aleatórios da população (sem reposição dentro do torneio)
2. O competidor com **menor fitness** (menor custo) vence e se torna um pai
3. Repete independentemente para o segundo pai

### Efeito do parâmetro `tournament_size` (K)

| K | Pressão de seleção | Diversidade |
|---|---|---|
| 2 | Baixa — qualquer um pode vencer | Alta |
| 3 (padrão) | Moderada | Moderada |
| N (tamanho da população) | Máxima — sempre o melhor vence | Muito baixa |

### Vantagem
- **Nenhuma normalização de fitness necessária** — funciona diretamente com valores brutos
- Fácil de controlar a pressão via `tournament_size`
- Eficiente computacionalmente (`O(K)` por seleção)

### Desvantagem
- Com K muito grande, converge rapidamente para explorar apenas os melhores indivíduos
- O mesmo indivíduo pode ser selecionado como os dois pais

In [14]:
# A. Implementação "Bare-metal"
import random

sim_population = list(range(5))   # índices 0..4 representando os indivíduos
sim_fitness    = [320.0, 180.0, 270.0, 410.0, 230.0]

def run_tournament(population, fitness, k, fixed_competitors=None):
    """Executa um torneio de tamanho k e retorna o índice vencedor."""
    competitors = fixed_competitors if fixed_competitors else random.sample(population, k)
    winner = min(competitors, key=lambda i: fitness[i])
    print(f"  Competidores: {competitors} → fitness: {[fitness[c] for c in competitors]}")
    print(f"  Vencedor: Ind {winner} (custo={fitness[winner]:.0f})")
    return winner

# Torneios com K=3 (fixos para ser reproduzível no notebook)
print("=== Torneio K=3 ===")
print("Torneio para Pai 1:")
p1 = run_tournament(sim_population, sim_fitness, k=3, fixed_competitors=[0, 2, 4])
print("Torneio para Pai 2:")
p2 = run_tournament(sim_population, sim_fitness, k=3, fixed_competitors=[1, 3, 4])
print(f"\nPais selecionados: Ind {p1} e Ind {p2}")

# Comparação: K=2 (baixa pressão) vs K=4 (alta pressão) — simulação estatística
print("\n=== Comparação de pressão de seleção (1000 torneios cada) ===")
for k in [2, 4]:
    wins = {i: 0 for i in sim_population}
    for _ in range(1000):
        competitors = random.sample(sim_population, k)
        winner = min(competitors, key=lambda i: sim_fitness[i])
        wins[winner] += 1
    print(f"\nK={k}:")
    for i, count in wins.items():
        bar = "█" * (count // 10)
        print(f"  Ind {i} (custo={sim_fitness[i]:.0f}): {count:4} vezes  {bar}")

=== Torneio K=3 ===
Torneio para Pai 1:
  Competidores: [0, 2, 4] → fitness: [320.0, 270.0, 230.0]
  Vencedor: Ind 4 (custo=230)
Torneio para Pai 2:
  Competidores: [1, 3, 4] → fitness: [180.0, 410.0, 230.0]
  Vencedor: Ind 1 (custo=180)

Pais selecionados: Ind 4 e Ind 1

=== Comparação de pressão de seleção (1000 torneios cada) ===

K=2:
  Ind 0 (custo=320):  103 vezes  ██████████
  Ind 1 (custo=180):  383 vezes  ██████████████████████████████████████
  Ind 2 (custo=270):  212 vezes  █████████████████████
  Ind 3 (custo=410):    0 vezes  
  Ind 4 (custo=230):  302 vezes  ██████████████████████████████

K=4:
  Ind 0 (custo=320):    0 vezes  
  Ind 1 (custo=180):  807 vezes  ████████████████████████████████████████████████████████████████████████████████
  Ind 2 (custo=270):    0 vezes  
  Ind 3 (custo=410):    0 vezes  
  Ind 4 (custo=230):  193 vezes  ███████████████████


---
### Uso com `TournamentSelectionStrategy` no repositório

O construtor recebe `tournament_size` (padrão = 3, mínimo = 2). Cada torneio usa `random.sample()` sem reposição, e o vencedor é determinado com `min(..., key=fitness_function)`.

In [17]:
# B. Usando a classe de domínio do projeto
tournament = TournamentSelectionStrategy(tournament_size=3)

parent1, parent2 = tournament.select_parents(population, evaluated_population, fitness_fn)
print("Parent 1:", names(parent1))
print("Parent 2:", names(parent2))

Parent 1: [['Origin', 'B', 'D', 'F', 'H', 'G', 'E', 'C', 'A']]
Parent 2: [['Origin', 'C', 'A', 'E', 'G', 'B', 'D', 'F', 'H']]
